Fine-tuning модели на задаче QA

In [ ]:
import os
import json
import random
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import gc

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507" #Qwen/Qwen3-4B-Instruct-2507
MAX_SEQ_LENGTH = 1024
OUTPUT_DIR = "./qwen_qa_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Загрузка базовой модели

In [ ]:
print("Загрузка модели без quantization...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    load_in_8bit=False,
)

tokenizer.pad_token = tokenizer.eos_token
FastLanguageModel.for_training(model)

Загрузка QA‑датасетов из JSONL

In [ ]:
def load_qa_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

train_path = "./github_qa_train.jsonl"
test_path = "./github_qa_test.jsonl"

In [ ]:
train_data = load_qa_jsonl(train_path)
test_data = load_qa_jsonl(test_path)

print(f"Train: {len(train_data)} примеров")
print(f"Test:  {len(test_data)} примеров")

Подготовка формата для SFT

In [ ]:
system_prompt = "Ты - эксперт по вопросам и ответам в IT-домене. Ответь на вопрос коротко и по делу."

def prepare_qa_dataset(data, tokenizer):
    formatted = []
    for item in data:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": item["input"]},
            {"role": "assistant", "content": item["output"]},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        formatted.append({"text": text})
    return Dataset.from_list(formatted)

In [ ]:
train_dataset = prepare_qa_dataset(train_data, tokenizer)

Настройка LoRA и SFTTrainer

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=4,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=RANDOM_SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "ft_checkpoints"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    num_train_epochs=1,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=1,
    remove_unused_columns=False,
    report_to="none",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
)

Настройка SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=True,
    format_prompt_fn=lambda x: x,
)

Обучение

In [ ]:
#torch.cuda.empty_cache()

In [ ]:
trainer.train()

Сохранение ЧИСТОГО LoRA‑адаптера для QA‑навыка

In [ ]:
lora_path = os.path.join(OUTPUT_DIR, "qa_lora_adapter")
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)